In [4]:
import numpy as np
import pandas as pd
import os
import psycopg
from dotenv import load_dotenv

CARGA A POSTGRES

In [ ]:

load_dotenv()
API_KEY = os.environ.get("API_KEY")
POSTGRESQL_URL_KEY = os.environ.get("POSTGRESQL_URL_KEY")


In [ ]:
# Los datos de la version para postgres (con None en lugar de NaN) 
# se guardan en el archivo  - carga_eda_modl.ipynb
# -> df_postgres = df_limpio.astype(object).where(pd.notna(df_limpio), None)
# -> df_postgres.to_pickle("df_postgres.pkl")
# Y los podemos llamar aqui:
df_postgres = pd.read_pickle(r"C:\Users\Kirsay\Desktop\AEMET_proyecto_local\data\df_postgres.pkl")



Definimos funcion

In [ ]:
# Carga por fecha + indicativo

def cargar_a_postgres(df_limpio: pd.DataFrame) -> None:
    """
    Crea (si no existe) la tabla all_stations_measurements y hace un
    upsert de las filas de df_limpio: inserta las nuevas y actualiza
    las existentes cuando ya hay una fila con la misma (fecha, indicativo).

    IMPORTANTE: df_limpio debe tener los NaN/NaT convertidos a None
    antes de llamar a esta función, por ejemplo:
        df_para_sql = df_limpio.astype(object).where(pd.notna(df_limpio), None)
    """
    conn_str = "postgresql://" + POSTGRESQL_URL_KEY

    with psycopg.connect(conn_str) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                CREATE TABLE IF NOT EXISTS all_stations_measurements (
                    fecha DATE,
                    indicativo CHAR(5),
                    nombre VARCHAR,
                    provincia VARCHAR,
                    altitud INTEGER,
                    tmed FLOAT,
                    prec FLOAT,
                    tmin FLOAT,
                    horatmin TIME,
                    tmax FLOAT,
                    horatmax TIME,
                    hrMedia INTEGER,
                    hrMax INTEGER,
                    horaHrMax TEXT,
                    hrMin INTEGER,
                    horaHrMin TEXT,
                    pintMax FLOAT,
                    dir INTEGER,
                    velmedia FLOAT,
                    racha FLOAT,
                    horaracha TEXT,
                    presMax FLOAT,
                    horaPresMax TEXT,
                    presMin FLOAT,
                    horaPresMin TEXT,
                    sol FLOAT,
                    horaPIntMax TEXT,
                    horatmin_son_varias_h BOOL,
                    horatmax_son_varias_h BOOL,
                    horaHrMax_son_varias_h BOOL,
                    horaHrMin_son_varias_h BOOL,
                    horaracha_son_varias_h BOOL,
                    horaPIntMax_son_varias_h BOOL,
                    CONSTRAINT fecha_indicativo_unique UNIQUE (fecha, indicativo)
                );
            """)

            cur.executemany(
                """
                INSERT INTO all_stations_measurements(
                    fecha, indicativo, nombre, provincia, altitud, tmed, prec,
                    tmin, horatmin, tmax, horatmax, hrMedia, hrMax, horaHrMax,
                    hrMin, horaHrMin, pintMax, dir, velmedia, racha, horaracha,
                    presMax, horaPresMax, presMin, horaPresMin, sol, horaPIntMax,
                    horatmin_son_varias_h, horatmax_son_varias_h,
                    horaHrMax_son_varias_h, horaHrMin_son_varias_h,
                    horaracha_son_varias_h, horaPIntMax_son_varias_h
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
                        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
                        %s, %s, %s, %s, %s)
                ON CONFLICT (fecha, indicativo) DO UPDATE SET
                    nombre = EXCLUDED.nombre,
                    provincia = EXCLUDED.provincia,
                    altitud = EXCLUDED.altitud,
                    tmed = EXCLUDED.tmed,
                    prec = EXCLUDED.prec,
                    tmin = EXCLUDED.tmin,
                    horatmin = EXCLUDED.horatmin,
                    tmax = EXCLUDED.tmax,
                    horatmax = EXCLUDED.horatmax,
                    hrMedia = EXCLUDED.hrMedia,
                    hrMax = EXCLUDED.hrMax,
                    horaHrMax = EXCLUDED.horaHrMax,
                    hrMin = EXCLUDED.hrMin,
                    horaHrMin = EXCLUDED.horaHrMin,
                    pintMax = EXCLUDED.pintMax,
                    dir = EXCLUDED.dir,
                    velmedia = EXCLUDED.velmedia,
                    racha = EXCLUDED.racha,
                    horaracha = EXCLUDED.horaracha,
                    presMax = EXCLUDED.presMax,
                    horaPresMax = EXCLUDED.horaPresMax,
                    presMin = EXCLUDED.presMin,
                    horaPresMin = EXCLUDED.horaPresMin,
                    sol = EXCLUDED.sol,
                    horaPIntMax = EXCLUDED.horaPIntMax,
                    horatmin_son_varias_h = EXCLUDED.horatmin_son_varias_h,
                    horatmax_son_varias_h = EXCLUDED.horatmax_son_varias_h,
                    horaHrMax_son_varias_h = EXCLUDED.horaHrMax_son_varias_h,
                    horaHrMin_son_varias_h = EXCLUDED.horaHrMin_son_varias_h,
                    horaracha_son_varias_h = EXCLUDED.horaracha_son_varias_h,
                    horaPIntMax_son_varias_h = EXCLUDED.horaPIntMax_son_varias_h;
                """,
                df_limpio.itertuples(index=False, name=None),
            )

        conn.commit()



In [ ]:
cargar_a_postgres(df_postgres)